[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees/cours/seance3_cours.ipynb)

# Séance 2.3 — Agréger et croiser plusieurs tables

**Cours** · durée : 2h — cinq techniques, chacune suivie de deux exercices

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- filtrer sur plusieurs conditions sans se noyer dans les parenthèses
- classer et extraire un top 5 en une commande
- répondre à « combien par ... ? » avec `groupby`
- calculer plusieurs indicateurs d'un coup avec `agg`
- rassembler trois fichiers en une seule table avec `merge`
- croiser deux dimensions avec un tableau croisé

## Retour à la question de départ

> *« Sur quel marché faut-il investir l'an prochain ? »*

Vous savez maintenant charger et nettoyer. Et pourtant vous ne pouvez toujours
pas répondre — pour une raison très simple :

**`ventes.csv` ne contient pas le pays.** Il contient un `client_id`. Le pays
est dans `clients.csv`.

C'est la situation normale en entreprise : l'information est **répartie entre
plusieurs fichiers**, et la réponse naît de leur croisement. C'est l'objet de
cette séance.

> 📋 **Comment on travaille aujourd'hui.** Cinq techniques. Pour chacune : une
> démonstration que vous suivez, puis **deux exercices que vous faites** — le
> premier a des `____` à remplir, le second est une cellule vide où vous
> écrivez tout. La cellule de vérification vous dit immédiatement si votre
> réponse est bonne.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")       ## une ligne = un produit
clients = pd.read_csv(BASE + "clients.csv")     ## une ligne = un client
produits = pd.read_csv(BASE + "produits.csv")   ## une ligne = une reference

ventes["ca"] = ventes["qte"] * ventes["prix"]   ## le CA de chaque ligne
print(ventes.shape, clients.shape, produits.shape)

## 1. Filtrer sur plusieurs conditions

### Empiler avec `and` / `or`

In [ ]:
grosses = ventes.query("qte >= 50 and prix < 2")   ## les deux a la fois
print(len(grosses), "lignes : beaucoup d'unites, prix unitaire faible")

Vous verrez souvent l'autre écriture :

```python
ventes[(ventes["qte"] >= 50) & (ventes["prix"] < 2)]
```

Comptez : **6 crochets, 4 parenthèses, 3 fois le nom de la table** — contre
une paire de parenthèses et une paire de guillemets avec `.query()`.
Reconnaissez cette écriture quand vous la croisez dans du code trouvé en
ligne, mais écrivez `.query()`.

### Une liste de valeurs — `in`

In [ ]:
# Attention aux guillemets : doubles a l'exterieur, simples a l'interieur
sud = clients.query("pays in ['Espagne', 'Portugal', 'Italie']")   ## in
print(len(sud), "clients dans ces trois pays")

### Un intervalle

In [ ]:
moyennes = ventes.query("50 <= qte <= 100")   ## un encadrement
print(len(moyennes), "lignes")

---

### ✏️ À vous 1a — Trois pays d'un coup

> **Votre mission :**
> - Compter les clients situés en France, en Allemagne ou en Belgique → `nb_ue`.
> - ⚠️ Guillemets doubles à l'extérieur, simples à l'intérieur.

In [ ]:
nb_ue = len(clients.query("pays ____ ['France', 'Allemagne', 'Belgique']"))

print(nb_ue)

In [ ]:
verifier("1a - clients dans trois pays", nb_ue == 121,
         "le mot-cle est in, et les noms de pays vont entre guillemets simples")

---

### ✏️ À vous 1b — Beaucoup d'unités, petit prix

> **Votre mission :**
> - Le service achats cherche les lignes qui partent **en volume à bas prix** : entre 50 et 100 unités, à moins de 2 € l'unité.
> - Combien y en a-t-il ? → `nb_volume`
> - Tout est à écrire : une seule condition `query`, un encadrement **et** un `and`.

In [ ]:
verifier("1b - volume a bas prix", nb_volume == 826,
         "un encadrement 50 <= qte <= 100, puis and prix < 2, dans la meme chaine")

## 2. Trier et extraire un top

In [ ]:
# ascending=False : du plus grand au plus petit
ventes.sort_values("prix", ascending=False).head(3)[["prod_id", "qte", "prix"]]

In [ ]:
# nlargest fait la meme chose en plus court
ventes.nlargest(3, "prix")[["prod_id", "prix"]]   ## trie + head, d'un coup

---

### ✏️ À vous 2a — Les plus grosses quantités

> **Votre mission :**
> - Afficher les **5 lignes** où la quantité commandée est la plus élevée → `grosses_qte`.
> - Mettre la quantité maximale dans `qte_max`.

In [ ]:
grosses_qte = ventes.____(5, "____")
qte_max = grosses_qte["qte"].max()

print(qte_max)
grosses_qte[["prod_id", "qte", "prix"]]

In [ ]:
verifier("2a - quantite maximale", qte_max == 1440,
         "nlargest prend deux arguments : combien de lignes, et sur quelle colonne")

---

### ✏️ À vous 2b — Les cinq plus grosses lignes de chiffre d'affaires

> **Votre mission :**
> - Classer les lignes par **chiffre d'affaires** décroissant et garder les cinq premières.
> - Mettre le `prod_id` de la toute première dans `prod_top`, et afficher les cinq lignes.
> - Regardez ce `prod_id`. Est-ce un produit ?

In [ ]:
verifier("2b - la plus grosse ligne", prod_top == "M",
         "nlargest(5, 'ca'), puis .iloc[0] pour la premiere ligne")

## 3. `groupby` — la commande la plus utile de tout le bloc

`groupby` répond à toutes les questions de la forme **« combien par ... ? »**.

Trois temps, toujours les mêmes :

1. **Découper** les lignes en paquets selon une colonne
2. **Calculer** un indicateur dans chaque paquet
3. **Recoller** les résultats en un tableau

In [ ]:
# "Combien de chiffre d'affaires par client ?"
ca_client = ventes.groupby("client_id")["ca"].sum()   ## decouper, calculer, recoller

ca_client.nlargest(5).round(2)   ## les cinq plus gros clients

Un client pèse à lui seul **143 825 €**. Gardez ce chiffre en tête, on y
reviendra en séance 2.4.

### Plusieurs indicateurs d'un coup — `agg`

In [ ]:
resume = ventes.groupby("client_id").agg(
    ca=("ca", "sum"),              # total depense
    nb_lignes=("cmd_id", "count"), # nombre de LIGNES
    nb_cmd=("cmd_id", "nunique"),  # nombre de COMMANDES distinctes
)
resume.nlargest(3, "ca").round(2)

La syntaxe se lit : `nom_voulu=("colonne_source", "operation")`.

> ⚠️ **`count` vs `nunique` — l'erreur classique.**
> `count` compte les **lignes**. `nunique` compte les **valeurs distinctes**.
> Une commande de 30 articles occupe 30 lignes mais reste **une** commande.
> Regardez l'écart entre `nb_lignes` et `nb_cmd` ci-dessus : confondre les
> deux, c'est diviser son panier moyen par 20.

Les opérations disponibles : `"sum"`, `"mean"`, `"median"`, `"min"`, `"max"`,
`"count"`, `"nunique"`, `"std"`.

---

### ✏️ À vous 3a — Le panier moyen

> **Votre mission :**
> - Un **panier**, c'est une commande entière — pas une ligne.
> - Calculer le CA total de chaque commande → `par_cmd`, puis le nombre de commandes → `nb_cmd_total` et le panier moyen arrondi à 2 décimales → `panier_moyen`.

In [ ]:
par_cmd = ventes.groupby("____")["ca"].sum()

nb_cmd_total = len(par_cmd)
panier_moyen = round(par_cmd.____(), 2)

print(nb_cmd_total, "commandes | panier moyen :", panier_moyen)

In [ ]:
verifier("3a - nombre de commandes", nb_cmd_total == 1955,
         "groupby('cmd_id') : une ligne du resultat = une commande")
verifier("3b - panier moyen", panier_moyen == 589.73,
         "mean() sur le CA par commande, pas sur le CA par ligne")

---

### ✏️ À vous 3b — Le meilleur client, vraiment ?

> **Votre mission :**
> - Construire `profil` : une ligne par client, avec son CA total (`ca`) et son nombre de **commandes distinctes** (`nb_cmd`).
> - Ajouter une colonne `panier` = CA ÷ nombre de commandes.
> - Afficher le client au plus gros panier moyen. Puis le meilleur **parmi ceux qui ont au moins 5 commandes** → mettre son identifiant dans `client_fidele`.
> - La réponse change. Lequel des deux présenteriez-vous à un directeur commercial ?

In [ ]:
verifier("3c - client fidele au meilleur panier", client_fidele == 12753,
         "filtrez sur nb_cmd >= 5 AVANT de classer par panier")

## 4. `merge` — rassembler les fichiers

C'est l'équivalent du `RECHERCHEV` d'Excel, en beaucoup plus sûr.

Les deux tables ont une colonne en commun : `client_id`. `merge` s'en sert
pour aller chercher, pour chaque vente, les informations du client
correspondant.

In [ ]:
avant = len(ventes)
vc = ventes.merge(clients, on="client_id")   ## on = la colonne commune

# LE reflexe : verifier qu'on n'a ni perdu ni duplique de lignes
print(avant, "->", len(vc))

> ⚠️ **Ne sautez jamais cette vérification.** Si la clé de jointure n'est pas
> unique dans la table de droite, `merge` **duplique** des lignes sans rien
> dire. Vos totaux deviennent faux et rien ne vous alerte. Deux nombres
> affichés, une seconde de lecture, et vous êtes tranquille.

`vc` contient maintenant les colonnes des deux tables :

In [ ]:
# pays et segment viennent de clients, qte et prix de ventes
vc[["client_id", "pays", "segment", "qte", "prix", "ca"]].head(3)

### Et enfin, la réponse à la question du bloc

In [ ]:
ca_pays = vc.groupby("pays")["ca"].sum().sort_values(ascending=False)

ca_pays.head(5).round(2)   ## le classement des marches

Le Royaume-Uni domine — c'est le marché domestique, sans surprise.

**Mais regardez l'Irlande : 261 205 €, deuxième marché du groupe.** Combien
de clients irlandais y a-t-il ?

In [ ]:
vc.query("pays == 'Irlande'")["client_id"].nunique()   ## combien ?

**Deux.** Deux clients suffisent à faire le deuxième marché du groupe. Vous
allez chiffrer ce que ça représente dans l'exercice 4b.

---

### ✏️ À vous 4a — Le panier moyen par pays

> **Votre mission :**
> - À partir de `vc` : pour chaque pays, le CA total (`ca`) et le nombre de **commandes distinctes** (`nb_cmd`).
> - Ajouter une colonne `panier` = CA ÷ nombre de commandes, arrondie à 2 décimales.
> - Mettre le panier moyen irlandais dans `panier_irl`.

In [ ]:
parpays = vc.groupby("pays").agg(
    ca=("ca", "sum"),
    nb_cmd=("cmd_id", "____"),
)
parpays["panier"] = (parpays["ca"] / parpays["____"]).round(2)

panier_irl = parpays.loc["Irlande", "panier"]
print(panier_irl)

In [ ]:
verifier("4a - panier moyen irlandais", panier_irl == 1020.33,
         "avec count au lieu de nunique le panier serait ridiculement bas")

---

### ✏️ À vous 4b — Chiffrer l'anomalie irlandaise

> **Votre mission :**
> - Quelle **part du chiffre d'affaires total** l'Irlande représente-t-elle, en % arrondi à 1 décimale ? → `part_irl`
> - Combien de clients irlandais y a-t-il ? → `nb_cli_irl`
> - Vous disposez déjà de `ca_pays` et de `vc`. Regardez les deux chiffres ensemble : que diriez-vous à un dirigeant ?

In [ ]:
verifier("4b - part de l'Irlande", part_irl == 22.7,
         "divisez le CA irlandais par ca_pays.sum()")
verifier("4c - clients irlandais", nb_cli_irl == 2,
         "nunique() sur client_id apres avoir filtre sur l'Irlande")

## 5. Croiser deux dimensions

Avant de croiser, ajoutons les informations produit — une deuxième jointure,
sur une autre clé :

In [ ]:
complet = vc.merge(produits, on="prod_id")   ## deuxieme jointure, autre cle
print(len(complet), "lignes,", complet.shape[1], "colonnes")   ## toujours verifier

complet.groupby("categorie")["ca"].sum().nlargest(3).round(2)   ## le trio de tete

`groupby` répond à « par pays ». Et « par pays **et** par segment » ?

In [ ]:
top4 = ca_pays.head(4).index   ## les etiquettes, pas les valeurs

vc.query("pays in @top4").pivot_table(
    values="ca",          ## ce qu'on calcule
    index="pays",         ## ce qui va en lignes
    columns="segment",    ## ce qui va en colonnes
    aggfunc="sum",        ## comment on l'agrege
).round(0)

> 💡 Le `@top4` dans `query()` veut dire « va chercher la variable Python
> nommée `top4` ». Pratique pour ne pas retaper une liste.

Deux cases sont vides pour l'Irlande. Ce n'est pas un bug : ses deux clients
sont tous deux classés « premium », il n'y a donc **aucune** ligne
« occasionnel » ou « standard » à additionner. **Une case vide dans un
tableau croisé est une information**, pas une erreur.

### Compter plutôt que sommer — `crosstab`

In [ ]:
pd.crosstab(complet.query("pays in @top4")["pays"],
            complet.query("pays in @top4")["categorie"])

---

### ✏️ À vous 5a — Le tableau croisé complet

> **Votre mission :**
> - Croiser `pays` (en lignes) et `segment` (en colonnes) sur **tous** les pays, avec la somme du `ca` → `tableau`.
> - Mettre le CA des clients « premium » français dans `fr_premium` (arrondi à 0 décimale).

In [ ]:
tableau = vc.pivot_table(values="ca", index="____", columns="____", aggfunc="sum")

fr_premium = round(tableau.loc["France", "premium"], 0)
print(fr_premium)

In [ ]:
verifier("5a - premium francais", fr_premium == 114433.0,
         "index=pays (lignes), columns=segment (colonnes), aggfunc='sum'")

---

### ✏️ À vous 5b — Le produit numéro un de chaque pays

> **Votre mission :**
> - À partir de `complet` : le CA par pays **et** par produit, trié par pays, et à l'intérieur de chaque pays du plus gros CA au plus petit.
> - Garder la **première ligne de chaque pays** et l'afficher. Le résultat devrait vous alerter.
> - Compter combien de pays ont `POST` en tête → `nb_post`.
> - *Nouveau :* `df.sort_values(['pays', 'ca'], ascending=[True, False])` — deux colonnes, deux sens ; et `df.groupby('pays').head(1)` prend la première ligne **de chaque groupe**.

In [ ]:
verifier("5b - pays dont le numero un est POST", nb_post == 12,
         "sort_values sur deux colonnes, puis groupby('pays').head(1)")

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| filtrer sur une condition | `df.query("prix > 10")` |
| combiner des conditions | `df.query("prix > 10 and pays == 'France'")` |
| une liste de valeurs | `df.query("pays in ['France', 'Belgique']")` |
| un intervalle | `df.query("50 <= qte <= 100")` |
| trier | `df.sort_values("ca", ascending=False)` |
| les 5 plus grands | `df.nlargest(5, "ca")` |
| total par groupe | `df.groupby("pays")["ca"].sum()` |
| plusieurs indicateurs | `df.groupby("pays").agg(ca=("ca", "sum"), n=("cmd_id", "nunique"))` |
| joindre deux tables | `a.merge(b, on="client_id")` |
| croiser deux dimensions | `df.pivot_table(values="ca", index="pays", columns="segment", aggfunc="sum")` |
| compter des croisements | `pd.crosstab(df["pays"], df["categorie"])` |

## Les deux erreurs à ne jamais commettre

1. **Faire un `merge` sans vérifier le nombre de lignes avant et après.**
   Un `merge` peut silencieusement dupliquer ou faire disparaître des lignes.
   `print(len(a), "->", len(fusion))` : une seconde, et vous dormez tranquille.

2. **Confondre `count` et `nunique`.** `count` compte les lignes,
   `nunique` compte les valeurs distinctes. Une commande de 30 articles,
   c'est 30 lignes mais **une** commande.